In [14]:
import numpy as np
import pandas as pd 
import cmath 
import scipy.special as sp
import scipy.signal as spg
import scipy.constants as const
import matplotlib.pyplot as plt
from scipy.constants import c, mu_0, epsilon_0, pi
from scipy.special import ai_zeros

I'm now looking at an alternative formulation to predict the location of TE and TM modes of the alumina disk, which essentially makes a modification to the following functions from Jackson's Classical Electrodynamics: 

TM modes: 

$$ 
\omega_{m,n,p} = \frac{c}{\sqrt{\mu\epsilon}} \sqrt{\frac{x_{m,n}^2}{R^2} + \frac{p^2 \pi^2}{d^2}}
$$

and TE modes: 

$$ 
\omega_{m,n,p} = \frac{c}{\sqrt{\mu\epsilon}} \sqrt{\frac{x\prime_{m,n}^2}{R^2} + \frac{p^2 \pi^2}{d^2}}
$$

which uses instead of Bessel functions, Airy functions, to account for the realities of a dielectric-air boundary and resonant modes with a system of containment/evanescent decay (turning point formulation), represented by: 

$$
f_{mnp} = \frac{1}{2\pi\sqrt{\mu\epsilon}} \sqrt{ \left( \frac{m+\alpha_n}{R} \right)^2 + \left( \frac{p\pi}{d} \right)^2 }
$$

The Airy root $\alpha_n$ is mode contingent; if it's a TM mode, it should use regular Ai($-\alpha_n^{TM}$), while in a TE mode it should use Ai'($\alpha_n^{TE}$). 


Building the function: 

In [15]:
pi

3.141592653589793

In [83]:
mu_0

1.25663706212e-06

In [84]:
epsilon_0

8.8541878128e-12

In [10]:
a, ap, ai, aip = ai_zeros(1)

In [11]:
print(a, ap, ai, aip)

[-2.33810741] [-1.01879297] [0.53565666] [0.70121082]


In [39]:
# get airy roots
alpha_1, alpha_prime_1, _, _, = ai_zeros(1)
alpha = (-1)*alpha_1
alpha_2 = (-1)*alpha_prime_1 

print(alpha, alpha_2)

[2.33810741] [1.01879297]


In [90]:
def Airy_freq_solver(mode_kind, m, p, mu_r = 1,epsilon_r = 9.7598758464, 
                     R = 0.076225, d = 0.02542957): 
                    
                    # get airy roots
                    alpha_1, alpha_prime_1, _, _, = ai_zeros(1)

                    # set material properties
                    mu = mu_0*mu_r
                    epsilon = epsilon_0*epsilon_r

                    # set TE or TM airy property
                    if mode_kind == 'TM': 
                        alpha = (-1)*alpha_1 # regular Airy 
                    if mode_kind == 'TE': 
                        alpha = (-1)*alpha_prime_1 # derivative Airy

                    # calculate omega_mnp
                    omega_mnp = ((c/(np.sqrt(mu_r*epsilon_r))) * np.sqrt(((m + alpha)/R)**2 + (p**2*pi**2)/d**2))
                    freq = omega_mnp/(2*np.pi) # Hz
                    freq_GHz = freq/(10**9) # GHz

                    return freq_GHz

In [91]:
test = Airy_freq_solver('TE', m = 10, p = 1)

In [92]:
test

array([2.90419618])

In [94]:
def get_TE_TM_m_p(N_m, p): 
    m_array = []
    TM_freqs = []
    TE_freqs = []
    for i in range(0, N_m): 
        TM_freq = Airy_freq_solver('TM', m = i, p=p)
        TE_freq = Airy_freq_solver('TE', m = i, p=p)
        m_array.append(i)
        TM_freqs.append(TM_freq)
        TE_freqs.append(TE_freq)
    eigenfreqs = {'m': m_array, 'TE': TE_freqs, 'TM': TM_freqs}
    eigen_m_table = pd.DataFrame(eigenfreqs)
    return eigen_m_table




In [99]:
test2 = get_TE_TM_m_p(N_m = 20, p = 1)

In [100]:
test2

,m,TE,TM
0,0,[1.8978260304565064],[1.9441045824498968]
1,1,[1.9296866667028159],[2.0018541575795266]
2,2,[1.9813958877231335],[2.0773997091068876]
3,3,[2.0514533235208785],[2.168882448875347]
4,4,[2.138056092750322],[2.274380057322184]
5,5,[2.2392853805113546],[2.392038932733619]
6,6,[2.3532543885989434],[2.520156326251434]
7,7,[2.4782060975656743],[2.6572199013706133]
8,8,[2.612565166465018],[2.8019170833909723]
9,9,[2.754955549279204],[2.9531259916602774]
